# Why a larger batch size produces more stable and less noisy gradients ?

[Link to the explanations](https://spurious-ruby-475.notion.site/Deep-Learning-3319a03e8f308097a4a7df8195bc5113#3319a03e8f308051954af809d4422601)

In [1]:
import numpy as np

class LinearRegression:
    def __init__(self, input_features: int, output_features: int, alpha: float = 1e-4) -> None:
        self.intput_features = input_features
        self.output_features = output_features
        self.alpha = alpha

        self.theta = np.random.randn(input_features, output_features) * 0.01
        self.gradient_history = []
        self.loss_history = []

    def forward(self, x: np.ndarray) -> np.ndarray:
        return x @ self.theta

    def _backpropagation(self, x: np.ndarray, y: np.ndarray, y_hat: np.ndarray) -> np.ndarray:
        dtheta = (1 / x.shape[0]) * x.T @ (y_hat - y)

        return dtheta

    def _update_theta(self, dtheta: np.ndarray) -> np.ndarray:
        self.theta -= self.alpha * dtheta

    def train(self, x: np.ndarray, y: np.ndarray, batch_size: int, total_steps: int, rng=None) -> None:
        n = x.shape[0]
        if rng is None:
            rng = np.random.default_rng()
        bs = min(batch_size, n)
        self.loss_history = []
        self.gradient_history = []
        log_every = max(1, total_steps // 400)
        for step in range(total_steps):
            idx = rng.choice(n, size=bs, replace=False)
            x_b = x[idx]
            y_b = y[idx]
            y_hat = self.forward(x=x_b)
            loss = (1 / (2 * bs)) * np.sum((y_b - y_hat) ** 2)
            dtheta = self._backpropagation(x=x_b, y=y_b, y_hat=y_hat)
            self._update_theta(dtheta=dtheta)
            if step % log_every == 0:
                self.loss_history.append(loss)
                self.gradient_history.append(dtheta.copy())
        

In [2]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

def generate_dataset(n: int, noise_std: float = 100.0) -> tuple[np.ndarray, np.ndarray]:
    a = np.random.uniform(low=-100, high=100)
    x1 = np.linspace(start=0, stop=100, num=n)
    x = np.column_stack([x1, x1**2, np.sin(x1 / 10)])
    x_std = x.std(axis=0)
    x_std = np.where(x_std < 1e-12, 1.0, x_std)
    x = (x - x.mean(axis=0)) / x_std
    noise = np.random.randn(n, 1) * noise_std
    y = a * x1.reshape(-1, 1) + noise
    y = (y - y.mean(axis=0)) / y.std(axis=0)
    return x, y


def mean_squared_gradient_error(
    x: np.ndarray,
    y: np.ndarray,
    theta: np.ndarray,
    batch_size: int,
    n_samples: int = 400,
    seed: int = 0,
) -> float:
    n = x.shape[0]
    y_hat = x @ theta
    g_full = (1 / n) * x.T @ (y_hat - y)
    rng = np.random.default_rng(seed)
    bs = min(batch_size, n)
    err = 0.0
    for _ in range(n_samples):
        idx = rng.choice(n, size=bs, replace=False)
        x_b, y_b = x[idx], y[idx]
        yh = x_b @ theta
        g_b = (1 / bs) * x_b.T @ (yh - y_b)
        d = g_b.ravel() - g_full.ravel()
        err += float(d @ d)
    return err / n_samples


x, y = generate_dataset(n=1000)
px.scatter(x=x[:, 0], y=y.ravel())

In [3]:
np.random.seed(42)
theta_probe = np.random.randn(3, 1) * 0.5
batch_sizes_probe = np.array([1, 2, 4, 8, 16, 32, 64, 128, 256, 500])
mse_grad = [mean_squared_gradient_error(x, y, theta_probe, int(bs)) for bs in batch_sizes_probe]

SMALL_BATCH_SIZE = 1
LARGE_BATCH_SIZE = 128
TOTAL_STEPS = 4000

theta0 = np.random.randn(3, 1) * 0.01
lr_1 = LinearRegression(input_features=3, output_features=1)
lr_1.theta = theta0.copy()
lr_1.train(x=x, y=y, batch_size=SMALL_BATCH_SIZE, total_steps=TOTAL_STEPS, rng=np.random.default_rng(12345))

lr_2 = LinearRegression(input_features=3, output_features=1)
lr_2.theta = theta0.copy()
lr_2.train(x=x, y=y, batch_size=LARGE_BATCH_SIZE, total_steps=TOTAL_STEPS, rng=np.random.default_rng(12345))

In [4]:
fig_grad = go.Figure()
fig_grad.add_trace(
    go.Scatter(
        x=batch_sizes_probe,
        y=mse_grad,
        mode='lines+markers',
        name='E[||g_hat - g_full||^2]',
    )
)
fig_grad.update_layout(
    title='Mini-batch gradient vs full-batch gradient (fixed θ)',
    xaxis_title='Batch size',
    yaxis_title='Mean squared error to full-batch gradient',
    yaxis_type='log',
    xaxis_type='log',
)
fig_grad.show()

fig_loss = go.Figure()
fig_loss.add_trace(go.Scatter(y=lr_1.loss_history, mode='lines', name=f'Batch {SMALL_BATCH_SIZE}'))
fig_loss.add_trace(go.Scatter(y=lr_2.loss_history, mode='lines', name=f'Batch {LARGE_BATCH_SIZE}'))
fig_loss.update_layout(
    title=f'Training batch loss (same initial θ, same total SGD steps = {TOTAL_STEPS})',
    xaxis_title='Logged step index',
    yaxis_title='Batch MSE (current mini-batch)',
)
fig_loss.show()

In [5]:
gh1 = np.stack([g.ravel() for g in lr_1.gradient_history])
gh2 = np.stack([g.ravel() for g in lr_2.gradient_history])
print(f"Var of gradient coords (small batch, logged steps): {gh1.var(axis=0).sum():.6f}")
print(f"Var of gradient coords (large batch, logged steps): {gh2.var(axis=0).sum():.6f}")

Var of gradient coords (small batch, logged steps): 1.300833
Var of gradient coords (large batch, logged steps): 0.060818
